# Flat Model
##### Establish fair baselines.
#### Experiment 1 — Flat Multiclass using RF (No Balancing)

#### Experiment 2 — Flat Multiclass (Class Weighting)
##### Random Forest + class_weight='balanced'

#### Experiment 3 — Flat Multiclass (Resampling)
##### Random Forest, Fixed Undersampling + SMOTE

#### Experiment 4 — Flat Multiclass (Hyperparameters Optimization)
##### Optuna introduction

### This shows imbalance impact. 

In [1]:
# Evaluation metrics

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

def evaluate_model(y_true, y_pred, model_name):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted")
    }

In [2]:
# Flat Multiclass using RF

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# columns that contain categorical data
categorical_cols = ['proto', 'service','state']  

# one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Spliting the encoded data into features and target
features_encoded = data_encoded.drop('attack_cat', axis=1)  
target = data_encoded['attack_cat']  
print(target.value_counts())

# Spliting the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_encoded, target, test_size=0.2, random_state=42)


# RandomForest classifier
rf_classifier = RandomForestClassifier(
   n_estimators=300,       
    max_depth=None,         
    random_state=42,
    #class_weight='balanced'
)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

results.append(
    evaluate_model(y_test, y_pred, "RF Baseline")
)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
Accuracy: 0.87666
Confusion Matrix:
[[  108    89   138   127    36     1     0     6     0     0]
 [   62    51    96   139    15     2     0    73     5     0]
 [  110   111  1157  1439   159    19     0   167    23     0]
 [   80   121  1192  6700   222    20     0   193    27     4]
 [   63    19   158   194  4344     4     0    66    19     0]
 [    2     4    36   147    23 11060     0     4     7     0]
 [    0     0     0     0     0     0 18110     0     0     0]
 [    8    68   179   293    61     3     0  2125     2     0]
 [    0     0     4    39    45     3     0     5   175     1]
 [    0     0     1    28     5     0     0     0     0     3]]
Classification Report:
                precision    reca

In [3]:
# Flat Multiclass using RF (class weighting)

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# columns that contain categorical data
categorical_cols = ['proto', 'service','state']  

# one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Spliting the encoded data into features and target
features_encoded = data_encoded.drop('attack_cat', axis=1)  
target = data_encoded['attack_cat']  
print(target.value_counts())


# Spliting the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_encoded, target, test_size=0.2, random_state=42)


# RandomForest classifier
rf_classifier = RandomForestClassifier(
   n_estimators=300,       
    max_depth=None,         
    random_state=42,
    class_weight='balanced'
)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

results.append(
    evaluate_model(y_test, y_pred, "RF Class Weight")
)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
Accuracy: 0.87524
Confusion Matrix:
[[  107    89   137   129    36     1     0     6     0     0]
 [   62    55    96   136    14     2     0    72     6     0]
 [  110   111  1135  1460   163    21     0   165    20     0]
 [   81   121  1182  6701   237    18     0   185    29     5]
 [   63    19   158   206  4333     4     0    67    17     0]
 [    2     4    36   164    22 11045     0     3     6     1]
 [    0     0     0     3     0     0 18107     0     0     0]
 [    9    68   179   313    60     3     0  2107     0     0]
 [    0     0     2    36    57     4     0     6   167     0]
 [    0     0     1    26     5     0     0     0     0     5]]
Classification Report:
                precision    reca

In [4]:
# Flat Multiclass (Resampling)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)

# One hot encoding
categorical_cols = ['proto', 'service', 'state']
data_encoded = pd.get_dummies(data, columns=categorical_cols)

features_encoded = data_encoded.drop('attack_cat', axis=1)
target = data_encoded['attack_cat']

X_train, X_test, y_train, y_test = train_test_split(
    features_encoded, target, test_size=0.2, random_state=42, stratify=target
)

target_count = 7000  # majority classes target count

# Compute the sampling strategy mathematically
counts = y_train.value_counts()
undersample_dict = {cls: min(count, target_count) for cls, count in counts.items()}

# Apply undersampling
rus = RandomUnderSampler(sampling_strategy=undersample_dict, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("After undersampling:")
print(pd.Series(y_resampled).value_counts())

# SMOTE up for minority classes
cat_indices = [2, 3, 4]  # categorical column indices
smote = SMOTE(sampling_strategy='not majority', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_resampled, y_resampled)

print("After SMOTE:")
print(pd.Series(y_resampled).value_counts())


# Model training
model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
model.fit(X_resampled, y_resampled)


# Performance evaluation
y_pred = model.predict(X_test)

results.append(
    evaluate_model(y_test, y_pred, "RF Resampled")
)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

After undersampling:
attack_cat
DoS               7000
Exploits          7000
Fuzzers           7000
Generic           7000
Normal            7000
Reconnaissance    7000
Analysis          2086
Backdoor          1810
Shellcode         1172
Worms              136
Name: count, dtype: int64
After SMOTE:
attack_cat
Analysis          7000
Backdoor          7000
DoS               7000
Exploits          7000
Fuzzers           7000
Generic           7000
Normal            7000
Reconnaissance    7000
Shellcode         7000
Worms             7000
Name: count, dtype: int64
Accuracy: 0.84774

Confusion Matrix:
[[  137   148   224    12     0     0     0     0     0     0]
 [   84   168   163    13     5     0     0    13     3     3]
 [  251   406  1957   373    53     3     0    47    58    19]
 [  444   527  2223  4405   221     4     1   429   161   220]
 [   91   196   233    37  4014     0     0    27   108    10]
 [    8    17    63   150    31 11116     0     2    18    10]
 [    0     0    

In [5]:
# Flat Multiclass with hyperparameter optimization

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import optuna


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)

categorical_cols = ['proto', 'service', 'state']

data_encoded = pd.get_dummies(data, columns=categorical_cols)

features = data_encoded.drop(['attack_cat', 'label'], axis=1)
target = data_encoded['attack_cat']

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)



# Objective function
def objective(trial):
    # Hyperparameters we aim to optimize
    target_count = trial.suggest_int("target_count", 1000, 15000)
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]

        X_val_fold = X_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]

        # Compute counts from THIS fold only
        counts = y_tr.value_counts()

        undersample_dict = {
            cls: min(count, target_count)
            for cls, count in counts.items()
        }

        # UnderSampling 
        rus = RandomUnderSampler(
            sampling_strategy=undersample_dict,
            random_state=42
        )

        X_res, y_res = rus.fit_resample(X_tr, y_tr)

        # SMOTE 
        smote = SMOTE(
            sampling_strategy="not majority",
            random_state=42
        )

        X_res, y_res = smote.fit_resample(X_res, y_res)

        # Random Classifier
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_res, y_res)

        preds = model.predict(X_val_fold)

        fold_scores.append(
            f1_score(y_val_fold, preds, average="macro")
        )

    return np.mean(fold_scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best target_count:", study.best_params)
print("Best F1:", study.best_value)

best_params = study.best_params
best_target_count = best_params["target_count"]

# Compute class counts on full training set
counts = y_train.value_counts()

undersample_dict = {
    cls: min(count, best_target_count)
    for cls, count in counts.items()
}
print("Performing resampling....................................")

# UnderSampling 
rus = RandomUnderSampler(
    sampling_strategy=undersample_dict,
    random_state=42
)

X_res, y_res = rus.fit_resample(X_train, y_train)

# SMOTE 
smote = SMOTE(
    
    sampling_strategy="not majority",
    random_state=42
)

X_res, y_res = smote.fit_resample(X_res, y_res)

final_model = RandomForestClassifier(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_res, y_res)

y_pred_optuna = final_model.predict(X_test)
print(classification_report(y_test, y_pred_optuna))
print(confusion_matrix(y_test, y_pred_optuna))

results.append(
    evaluate_model(y_test, y_pred_optuna, "RF Optuna Tuned")
)




C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-06 10:22:18,117] A new study created in memory with name: no-name-eec86355-1aa3-435d-851d-a25913996a3a
[I 2026-03-06 10:24:22,085] Trial 0 finished with value: 0.483526813237228 and parameters: {'target_count': 11601, 'n_estimators': 460, 'max_depth': 10, 'min_samples_split': 6}. Best is trial 0 with value: 0.483526813237228.
[I 2026-03-06 10:26:08,216] Trial 1 finished with value: 0.5605370599832257 and parameters: {'target_count': 13674, 'n_estimators': 103, 'max_depth': 28, 'min_samples_split': 6}. Best is trial 1 with value: 0.5605370599832257.
[I 2026-03-06 10:28:11,712] Trial 2 finished with value: 0.5148653638055086 and parameters: {

Best target_count: {'target_count': 13853, 'n_estimators': 317, 'max_depth': 30, 'min_samples_split': 3}
Best F1: 0.5633003035807915
Performing resampling....................................
                precision    recall  f1-score   support

      Analysis       0.11      0.21      0.14       521
      Backdoor       0.11      0.27      0.16       452
           DoS       0.37      0.64      0.47      3167
      Exploits       0.84      0.56      0.67      8635
       Fuzzers       0.59      0.85      0.70      4716
       Generic       1.00      0.97      0.99     11415
        Normal       0.99      0.83      0.90     18051
Reconnaissance       0.81      0.82      0.82      2716
     Shellcode       0.38      0.90      0.53       293
         Worms       0.17      0.71      0.28        34

      accuracy                           0.79     50000
     macro avg       0.54      0.68      0.56     50000
  weighted avg       0.86      0.79      0.81     50000

[[  112   127   243   

In [6]:
import pandas as pd

comparison_table = pd.DataFrame(results)
comparison_table = comparison_table.round(4)

print(comparison_table)

             Model  Accuracy  Macro Precision  Macro Recall  Macro F1  \
0      RF Baseline    0.8767           0.6225        0.5848    0.5938   
1  RF Class Weight    0.8752           0.6300        0.5863    0.5990   
2     RF Resampled    0.8477           0.5736        0.7124    0.5885   
3  RF Optuna Tuned    0.7937           0.5359        0.6768    0.5644   

   Weighted F1  
0       0.8757  
1       0.8742  
2       0.8621  
3       0.8128  
